# Part b): Writing our own neural network code

We fit the one-dimensional Runge function with our own feed-forward neural network (FFNN) and compare it with the best OLS model from project 1. The network classes live in `fys_stk4155_p2.nn`, while the data generation, optimizers and OLS code are imported from project 1 (`fys_stk4155_p1`, release v0.1.0).

In [ ]:
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scienceplots  # noqa: F401
from fys_stk4155_p1.data.design_matrix import univariate_polynomial_design_matrix
from fys_stk4155_p1.data.runge import generate_runge_data, runge_function
from fys_stk4155_p1.metrics import mean_squared_error, r2_score
from fys_stk4155_p1.optimization.optimizers import Adam, Optimizer, Plain, RMSProp
from fys_stk4155_p1.regression.degree_sweep import fit_polynomial_degree_sweep
from fys_stk4155_p1.regression.lasso import Lasso
from fys_stk4155_p1.regression.ordinary_least_squares import OLS
from fys_stk4155_p1.regression.ridge import Ridge
from joblib import Parallel, delayed
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler

from fys_stk4155_p2.nn.activations import Activation, LeakyReLU, ReLU, Sigmoid
from fys_stk4155_p2.nn.layers import Dense, Layer, Linear
from fys_stk4155_p2.nn.losses import MeanSquaredError, Regularization
from fys_stk4155_p2.nn.network import FFNN

plt.style.use(["science", "ieee"])

## Configuration

The data settings are those of the reference data set in project 1, so that the networks and OLS are compared on the same data and the same train-test split.

In [ ]:
# Data, identical to the reference data set of project 1
N_SAMPLES = 100
NOISE_STD = 0.1
SEED = 42
TEST_SIZE = 0.2

# Polynomial degree with the lowest test MSE on the reference split in project 1
OLS_DEGREE = 12

# Hyperparameter grid for the networks, searched with k-fold cross-validation
EPOCHS = [100, 1000, 10000]
BATCH_SIZE = [None, 8, 16, 32]  # None: full-batch gradient descent
LEARNING_RATE = [0.1, 0.01, 0.001, 0.0001]
OPTIMIZER = [Plain, RMSProp, Adam]
N_FOLDS = 5

# Cached grid-search results; recomputed automatically if the grid above changes,
# but not if the model code changes: delete the file to rerun the search then
GRID_SEARCH_PATH = Path("../data/results/b_grid_search.csv")

## Data

We sample the Runge function with Gaussian noise and split the samples into training and test data.

The network takes the raw $x$ as its only input feature, so the design matrix is the single column $x$ rather than the polynomial expansion used in project 1: the hidden layers learn the nonlinear features themselves. We standardize $x$ with the mean and standard deviation of the training data only, so that no information from the test data enters the training.

In [ ]:
x, y = generate_runge_data(n=N_SAMPLES, noise_std=NOISE_STD, seed=SEED)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=TEST_SIZE, random_state=SEED)

scaler = StandardScaler().fit(x_train.reshape(-1, 1))
X_train = scaler.transform(x_train.reshape(-1, 1))
X_test = scaler.transform(x_test.reshape(-1, 1))

## OLS baseline from project 1

We refit OLS with the polynomial degree selected in project 1. `fit_polynomial_degree_sweep` builds the polynomial design matrix, splits it with the same seed and test size as above (giving the same training and test samples), and standardizes the features on the training data.

In [ ]:
ols_results = fit_polynomial_degree_sweep(x, y, [OLS_DEGREE], OLS, test_size=TEST_SIZE, seed=SEED)
print(
    f"OLS (p = {OLS_DEGREE}): test MSE = {ols_results['mse_test'][0]:.5f}, "
    f"test R2 = {ols_results['r2_test'][0]:.4f}"
)

## Feed-forward neural networks

`create_ffnn` builds a network with sigmoid activations in the hidden layers and a linear output layer, since the target is an unbounded real number. The weights are drawn from a normal distribution whose width depends on the number of inputs to the layer (Section 8.10 of the lecture notes). For the sigmoid we use Xavier initialization, $\sigma_W^2 = 2/(n_\text{in} + n_\text{out})$, which keeps the pre-activations near zero, where the sigmoid is close to linear and its derivative is largest; the He initialization $\sigma_W^2 = 2/n_\text{in}$ is meant for the ReLU family (part c), which zeroes about half of its inputs. The biases are initialized to zero, as the random weights already break the symmetry between the neurons.

In [ ]:
# Weight initialization of the hidden layers, matched to their activation (Section 8.10)
INITIALIZATION = {Sigmoid: "xavier", ReLU: "he", LeakyReLU: "he"}


def create_ffnn(
    hidden_layers: list[int],
    optimizer: Optimizer,
    activation: type[Activation] = Sigmoid,
    regularization: Regularization = "none",
    lambda_: float = 0.0,
    *,
    rng: np.random.Generator,
) -> FFNN:
    """Build an FFNN for the one-dimensional Runge function, trained with the MSE.

    Args:
        hidden_layers: Number of neurons in each hidden layer.
        optimizer: Fresh optimizer instance, copied for every weight matrix and bias.
        activation: The activation function class which to use for the hidden layers.
        regularization: The regularization norm to apply, defaults to none.
        lambda_: The regularization strength to apply, defaults to 0.0.
        rng: Random number generator for the initial weights.

    Returns:
        The untrained network.
    """
    layers: list[Layer] = []
    n_inputs = 1  # x is the only input feature
    for n_neurons in hidden_layers:
        layers += [Dense(n_inputs, n_neurons, INITIALIZATION[activation], rng=rng), activation()]
        n_inputs = n_neurons
    layers += [Dense(n_inputs, 1, rng=rng), Linear()]

    return FFNN(layers, MeanSquaredError(), optimizer, regularization, lambda_)

## Hyperparameter search

We search over the number of epochs, the batch size, the learning rate and the optimizer for both architectures: plain gradient descent, and RMSprop and Adam. A batch size of `None` means full-batch training, so `Plain` with `None` is plain gradient descent, and the other batch sizes give stochastic gradient descent.

Each configuration is scored by its mean validation MSE over 5-fold cross-validation on the training data, as for the model selection in project 1, so that the test data is used only for the final comparison. Within each fold, $x$ is standardized with the statistics of that fold's training part. All configurations start from the same initial weights, so that differences come from the hyperparameters alone. A configuration whose predictions overflow is marked as diverged, with an infinite MSE.

The search trains about 2900 networks, so it runs in parallel on all cores and caches its results in `GRID_SEARCH_PATH`.

In [ ]:
architectures = {
    "1 hidden layer (50 nodes)": [50],
    "2 hidden layers (100 nodes)": [100, 100],
}
OPTIMIZERS_BY_NAME = {optimizer.__name__: optimizer for optimizer in OPTIMIZER}
GRID_COLUMNS = ["architecture", "optimizer", "learning_rate", "batch_size", "epochs"]


def as_batch_size(value: float) -> int | None:
    """Convert a batch size from the results table, where full-batch is missing, to fit's form."""
    return None if pd.isna(value) else int(value)


def cv_mse(
    architecture: str, optimizer: str, learning_rate: float, batch_size: int | None, epochs: int
) -> float:
    """Mean validation MSE over the k folds of the training data, or inf if training diverges."""
    fold_mse = []
    for fit_idx, val_idx in KFold(N_FOLDS, shuffle=True, random_state=SEED).split(x_train):
        fold_scaler = StandardScaler().fit(x_train[fit_idx].reshape(-1, 1))
        network = create_ffnn(
            hidden_layers=architectures[architecture],
            optimizer=OPTIMIZERS_BY_NAME[optimizer](learning_rate=learning_rate),
            rng=np.random.default_rng(SEED),
        )
        with np.errstate(over="ignore", invalid="ignore"):
            network.fit(
                fold_scaler.transform(x_train[fit_idx].reshape(-1, 1)),
                y_train[fit_idx],
                epochs=epochs,
                batch_size=batch_size,
            )
            prediction = network.predict(fold_scaler.transform(x_train[val_idx].reshape(-1, 1)))
        if not np.all(np.isfinite(prediction)):
            return np.inf
        fold_mse.append(mean_squared_error(y_train[val_idx], prediction))
    return float(np.mean(fold_mse))


grid = pd.DataFrame(
    product(architectures, OPTIMIZERS_BY_NAME, LEARNING_RATE, BATCH_SIZE, EPOCHS),
    columns=GRID_COLUMNS,
).astype({"batch_size": "Int64"})

cached = (
    pd.read_csv(GRID_SEARCH_PATH, dtype={"batch_size": "Int64"})
    if GRID_SEARCH_PATH.exists()
    else None
)
if cached is not None and cached[GRID_COLUMNS].equals(grid):
    grid_results = cached
else:
    scores = Parallel(n_jobs=-1, verbose=1)(
        delayed(cv_mse)(
            row.architecture,
            row.optimizer,
            row.learning_rate,
            as_batch_size(row.batch_size),
            int(row.epochs),
        )
        for row in grid.itertuples(index=False)
    )
    grid_results = grid.assign(cv_mse=scores)
    GRID_SEARCH_PATH.parent.mkdir(parents=True, exist_ok=True)
    grid_results.to_csv(GRID_SEARCH_PATH, index=False)

The best configuration for each combination of architecture and optimizer, sorted by the cross-validated MSE:

In [ ]:
best_configs = grid_results.loc[
    grid_results.groupby(["architecture", "optimizer"])["cv_mse"].idxmin()
].sort_values("cv_mse")
best_configs.round({"cv_mse": 5})

## Learning rates and optimizers

The cross-validated MSE as a function of the learning rate $\gamma$, for each optimizer with full-batch training (gradient descent, GD) and with minibatches (stochastic gradient descent, SGD). Each cell shows the best MSE over the number of epochs and, for SGD, the batch size. A cell is marked "div." if all its configurations diverged.

In [ ]:
METHOD_ORDER = [f"{opt}, {method}" for opt in OPTIMIZERS_BY_NAME for method in ("GD", "SGD")]
learning_rates = sorted(LEARNING_RATE)

lr_table = grid_results.assign(
    method=grid_results["optimizer"] + np.where(grid_results["batch_size"].isna(), ", GD", ", SGD")
).pivot_table(
    index=["architecture", "method"], columns="learning_rate", values="cv_mse", aggfunc="min"
)

finite = lr_table.to_numpy()[np.isfinite(lr_table.to_numpy())]
norm = LogNorm(vmin=finite.min(), vmax=finite.max())
cmap = plt.get_cmap("Blues").with_extremes(bad="#d3d1c7")

fig, axes = plt.subplots(
    1, len(architectures), figsize=(7, 3.4), sharey=True, constrained_layout=True
)
for ax, name in zip(axes, architectures, strict=True):
    table = lr_table.loc[name].loc[METHOD_ORDER, learning_rates].to_numpy()
    image = ax.imshow(np.ma.masked_invalid(table), cmap=cmap, norm=norm, aspect="auto")
    for (i, j), value in np.ndenumerate(table):
        if np.isfinite(value):
            text, color = f"{value:.4f}", "white" if norm(value) > 0.6 else "#1f1e1d"
        else:
            text, color = "div.", "#1f1e1d"
        ax.text(j, i, text, ha="center", va="center", fontsize=7, color=color)
    ax.set_xticks(
        range(len(learning_rates)), [f"$10^{{{round(np.log10(lr))}}}$" for lr in learning_rates]
    )
    ax.set_yticks(range(len(METHOD_ORDER)), METHOD_ORDER)
    ax.set_xlabel(r"Learning rate $\gamma$")
    ax.set_title(name, fontsize=9)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
fig.colorbar(image, ax=axes, label="Cross-validated MSE", shrink=0.9)
fig.savefig("../docs/figures/regression/learning_rates.pdf")
plt.show()

The heat map shows only the end point of training. To see how each optimizer gets there, we train the network with one hidden layer on all the training data with SGD (batch size 16) for every learning rate in the grid and record the training MSE after each epoch. Curves that stop have diverged.

In [ ]:
CURVE_ARCHITECTURE = "1 hidden layer (50 nodes)"
CURVE_BATCH_SIZE = 16
CURVE_EPOCHS = max(EPOCHS)


def training_curve(optimizer: str, learning_rate: float) -> list[float]:
    """Training MSE per epoch for the curve architecture, trained on all training data."""
    network = create_ffnn(
        hidden_layers=architectures[CURVE_ARCHITECTURE],
        optimizer=OPTIMIZERS_BY_NAME[optimizer](learning_rate=learning_rate),
        rng=np.random.default_rng(SEED),
    )
    with np.errstate(over="ignore", invalid="ignore"):
        return network.fit(X_train, y_train, epochs=CURVE_EPOCHS, batch_size=CURVE_BATCH_SIZE)


curve_configs = list(product(OPTIMIZERS_BY_NAME, learning_rates))
curves = dict(
    zip(
        curve_configs,
        Parallel(n_jobs=-1)(delayed(training_curve)(*config) for config in curve_configs),
        strict=True,
    )
)

# One sequential hue, light to dark with increasing learning rate
lr_shades = plt.get_cmap("Blues")(np.linspace(0.4, 1.0, len(learning_rates)))
lr_colors = dict(zip(learning_rates, lr_shades, strict=True))

fig, axes = plt.subplots(
    1, len(OPTIMIZERS_BY_NAME), figsize=(7, 3), sharey=True, constrained_layout=True
)
for ax, optimizer in zip(axes, OPTIMIZERS_BY_NAME, strict=True):
    for learning_rate in learning_rates:
        history = curves[optimizer, learning_rate]
        ax.plot(
            np.arange(1, len(history) + 1),
            history,
            lw=1.2,
            color=lr_colors[learning_rate],
            label=rf"$\gamma = 10^{{{round(np.log10(learning_rate))}}}$",
        )
    ax.axhline(NOISE_STD**2, color="black", ls=":", lw=0.8, label=r"noise floor $\sigma^2$")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(optimizer, fontsize=9)
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.3, lw=0.5)
axes[0].set_ylabel("Training MSE")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels, loc="outside lower center", ncols=len(labels), frameon=False, fontsize=8
)
fig.savefig("../docs/figures/regression/optimizer_curves.pdf")
plt.show()

## Training the selected networks

For each architecture, we retrain the configuration with the lowest cross-validated MSE on all the training data. These networks are compared with OLS below.

In [ ]:
best_per_architecture = grid_results.loc[
    grid_results.groupby("architecture")["cv_mse"].idxmin()
].set_index("architecture")

networks: dict[str, FFNN] = {}
histories: dict[str, list[float]] = {}

for name, hidden_layers in architectures.items():
    config = best_per_architecture.loc[name]
    print(config)
    network = create_ffnn(
        hidden_layers=hidden_layers,
        optimizer=OPTIMIZERS_BY_NAME[config.optimizer](learning_rate=config.learning_rate),
        rng=np.random.default_rng(SEED),
    )
    histories[name] = network.fit(
        X_train, y_train, epochs=int(config.epochs), batch_size=as_batch_size(config.batch_size)
    )
    networks[name] = network

## Comparison of the fitted models

We evaluate OLS and the two networks on a dense grid over $[-1, 1]$ and plot them on top of the Runge function and the data. OLS is refitted directly rather than taken from `ols_results`, since `fit_polynomial_degree_sweep` does not return the scaler of the polynomial features; with the same degree, training data and scaling, it is the same model. The networks were trained on standardized $x$, so the grid is passed through the same `scaler`.

In [ ]:
poly_scaler = StandardScaler().fit(univariate_polynomial_design_matrix(x_train, OLS_DEGREE))


def ols_design(x_values: np.ndarray) -> np.ndarray:
    """Standardized polynomial design matrix, scaled with the training statistics."""
    return poly_scaler.transform(univariate_polynomial_design_matrix(x_values, OLS_DEGREE))


ols = OLS().fit(ols_design(x_train), y_train)

x_plot = np.linspace(-1, 1, 500)
X_plot = scaler.transform(x_plot.reshape(-1, 1))
predictions = {"OLS": ols.predict(ols_design(x_plot))}
predictions |= {name: network.predict(X_plot) for name, network in networks.items()}

MODEL_LABELS = {"OLS": f"OLS ($p = {OLS_DEGREE}$)"} | {name: f"FFNN, {name}" for name in networks}

fig, ax = plt.subplots(figsize=(6, 4.6), constrained_layout=True)
ax.scatter(x_train, y_train, s=10, color="#b4b2a9", label="training data")
ax.scatter(
    x_test, y_test, s=14, facecolors="none", edgecolors="#5f5e5a", linewidths=0.8, label="test data"
)
ax.plot(x_plot, runge_function(x_plot), color="black", lw=1.5, label="Runge function")
for name, prediction in predictions.items():
    ax.plot(x_plot, prediction, lw=1.5, label=MODEL_LABELS[name])
ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.grid(alpha=0.3, lw=0.5)
fig.legend(loc="outside lower center", ncols=2, frameon=False, fontsize=8)
fig.savefig("../docs/figures/regression/model_comparison.pdf")
plt.show()

## Training history

The training MSE after each epoch, with the noise floor $\sigma^2$ as reference: even the true Runge function has an expected MSE of $\sigma^2$ on noisy data. A training MSE well below $\sigma^2$ means the network has started to fit the noise, while one far above it means it still underfits.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
for name, history in histories.items():
    ax.plot(
        np.arange(1, len(history) + 1),
        history,
        lw=1.5,
        label=MODEL_LABELS[name],
    )
ax.axhline(NOISE_STD**2, color="black", ls=":", lw=0.8, label=r"noise floor $\sigma^2$")
ax.set_xlabel("Epoch")
ax.set_xscale("log")
ax.set_ylabel("Training MSE")
ax.set_yscale("log")
ax.grid(alpha=0.3, lw=0.5)
ax.legend(frameon=False, fontsize=8)
fig.savefig("../docs/figures/regression/history.pdf")
plt.show()

## Metrics

MSE and $R^2$ on the training and test data, and the MSE against the true Runge function $f$ on the dense grid. The test set has only 20 samples, so its MSE is noisy and includes the irreducible error $\sigma^2$; the MSE against $f$ measures how well each model has learned the function itself. Note also that the test $R^2$ cannot approach 1 even for a perfect model: its expected value is at most about $1 - \sigma^2 / \mathrm{Var}(y_\text{test})$.

In [ ]:
# Each model as a function of the raw x, so all metrics use the same inputs
fitted = {"OLS": lambda x_values: ols.predict(ols_design(x_values))}
fitted |= {
    name: lambda x_values, net=network: net.predict(scaler.transform(x_values.reshape(-1, 1)))
    for name, network in networks.items()
}

metrics = pd.DataFrame(
    {
        MODEL_LABELS[name]: {
            "train MSE": mean_squared_error(y_train, predict(x_train)),
            "test MSE": mean_squared_error(y_test, predict(x_test)),
            "train R2": r2_score(y_train, predict(x_train)),
            "test R2": r2_score(y_test, predict(x_test)),
            "MSE vs f": mean_squared_error(runge_function(x_plot), predict(x_plot)),
        }
        for name, predict in fitted.items()
    }
).T
metrics.round(4)

# Part c) Testing different activation functions and depths of the neural network

This part asks us to (1) compare Sigmoid, ReLU and Leaky ReLU in the hidden layers, (2) study the result as a function of the number of hidden layers and nodes, and (3) look for overfitting.

First we configure the values to which we should sweep.

In [ ]:
N_HIDDEN_LAYERS = [1, 2, 4, 8]
N_NODES_PER_LAYER = [10, 20, 40, 80, 160]
ACTIVATION = [Sigmoid, ReLU, LeakyReLU]

# Training setup, fixed for all architectures: Adam with minibatches was among the best
# configurations in part b, and the least sensitive to the learning rate
C_OPTIMIZER = Adam
C_BATCH_SIZE = 32
C_EPOCHS = 5000
# A few learning rates per architecture, as deep ReLU networks may need smaller steps
C_LEARNING_RATE = [0.01, 0.001, 0.0001]

# Epochs for the training and validation curves, long enough for overfitting to show
C_CURVE_EPOCHS = 5000

# Cached architecture search, recomputed under the same rules as GRID_SEARCH_PATH
C_GRID_SEARCH_PATH = Path("../data/results/c_architecture_search.csv")

## Architecture search

Searching the full part b grid for all 60 architectures would be too expensive, so we fix the training setup to Adam with batch size 32 for 5000 epochs, and try only the learning rates in `C_LEARNING_RATE` for each architecture. A single fixed learning rate would favour the activations it happens to suit, while deep ReLU networks with He initialization may diverge where the sigmoid does not. `create_ffnn` matches the weight initialization to the activation.

Every architecture is scored with the same 5-fold cross-validation as in part b, starting from the same seed. We also record the mean MSE on each fold's training part, so the gap between the training and validation MSE can be used to detect overfitting. The search caches its results in `C_GRID_SEARCH_PATH`.

In [ ]:
ACTIVATIONS_BY_NAME = {activation.__name__: activation for activation in ACTIVATION}
C_GRID_COLUMNS = ["activation", "n_layers", "n_nodes", "learning_rate"]


def cv_train_val_mse(
    activation: str,
    n_layers: int,
    n_nodes: int,
    learning_rate: float,
    regularization: Regularization = "none",
    lambda_: float = 0.0,
) -> tuple[float, float]:
    """Mean training and validation MSE over the k folds, or inf for both if training diverges.

    The MSE is computed from the predictions, so it never includes the regularization
    penalty, and networks with different penalties are scored on equal terms.
    """
    train_mse, val_mse = [], []
    for fit_idx, val_idx in KFold(N_FOLDS, shuffle=True, random_state=SEED).split(x_train):
        fold_scaler = StandardScaler().fit(x_train[fit_idx].reshape(-1, 1))
        X_fold_fit = fold_scaler.transform(x_train[fit_idx].reshape(-1, 1))
        X_fold_val = fold_scaler.transform(x_train[val_idx].reshape(-1, 1))
        network = create_ffnn(
            hidden_layers=[n_nodes] * n_layers,
            optimizer=C_OPTIMIZER(learning_rate=learning_rate),
            activation=ACTIVATIONS_BY_NAME[activation],
            regularization=regularization,
            lambda_=lambda_,
            rng=np.random.default_rng(SEED),
        )
        with np.errstate(over="ignore", invalid="ignore"):
            network.fit(X_fold_fit, y_train[fit_idx], epochs=C_EPOCHS, batch_size=C_BATCH_SIZE)
            fit_prediction = network.predict(X_fold_fit)
            val_prediction = network.predict(X_fold_val)
        if not (np.all(np.isfinite(fit_prediction)) and np.all(np.isfinite(val_prediction))):
            return np.inf, np.inf
        train_mse.append(mean_squared_error(y_train[fit_idx], fit_prediction))
        val_mse.append(mean_squared_error(y_train[val_idx], val_prediction))
    return float(np.mean(train_mse)), float(np.mean(val_mse))


c_grid = pd.DataFrame(
    product(ACTIVATIONS_BY_NAME, N_HIDDEN_LAYERS, N_NODES_PER_LAYER, C_LEARNING_RATE),
    columns=C_GRID_COLUMNS,
)

c_cached = pd.read_csv(C_GRID_SEARCH_PATH) if C_GRID_SEARCH_PATH.exists() else None
if c_cached is not None and c_cached[C_GRID_COLUMNS].equals(c_grid):
    c_results = c_cached
else:
    c_scores = Parallel(n_jobs=-1, verbose=1)(
        delayed(cv_train_val_mse)(
            row.activation, int(row.n_layers), int(row.n_nodes), row.learning_rate
        )
        for row in c_grid.itertuples(index=False)
    )
    c_results = c_grid.assign(
        train_mse=[train for train, _ in c_scores], cv_mse=[val for _, val in c_scores]
    )
    C_GRID_SEARCH_PATH.parent.mkdir(parents=True, exist_ok=True)
    c_results.to_csv(C_GRID_SEARCH_PATH, index=False)

The best architecture for each activation, with its learning rate, mean training MSE and cross-validated MSE:

In [ ]:
def n_parameters(n_layers: int, n_nodes: int) -> int:
    """Number of weights and biases in a network with one input, one output and equal widths."""
    return 2 * n_nodes + (n_layers - 1) * (n_nodes + 1) * n_nodes + n_nodes + 1


def architecture_label(n_layers: int, n_nodes: int) -> str:
    """Short label for an architecture, e.g. 2 x 40 for two hidden layers of 40 nodes."""
    return rf"${n_layers} \times {n_nodes}$"


# The best learning rate for each architecture
c_best = c_results.loc[
    c_results.groupby(["activation", "n_layers", "n_nodes"])["cv_mse"].idxmin()
].assign(n_parameters=lambda df: n_parameters(df["n_layers"], df["n_nodes"]))

best_per_activation = (
    c_best.loc[c_best.groupby("activation")["cv_mse"].idxmin()]
    .set_index("activation")
    .loc[list(ACTIVATIONS_BY_NAME)]
)
best_per_activation.round({"train_mse": 5, "cv_mse": 5})

## Depth and width

The cross-validated MSE for every architecture and activation, each with its best learning rate. A cell is marked "div." if training diverged for both learning rates.

In [ ]:
c_tables = {
    activation: c_best[c_best["activation"] == activation]
    .pivot_table(index="n_layers", columns="n_nodes", values="cv_mse")
    .loc[N_HIDDEN_LAYERS, N_NODES_PER_LAYER]
    .to_numpy()
    for activation in ACTIVATIONS_BY_NAME
}

c_finite = np.concatenate([table[np.isfinite(table)] for table in c_tables.values()])
c_norm = LogNorm(vmin=c_finite.min(), vmax=c_finite.max())

fig, axes = plt.subplots(
    1, len(ACTIVATIONS_BY_NAME), figsize=(7, 2.8), sharey=True, constrained_layout=True
)
for ax, (activation, table) in zip(axes, c_tables.items(), strict=True):
    image = ax.imshow(np.ma.masked_invalid(table), cmap=cmap, norm=c_norm, aspect="auto")
    for (i, j), value in np.ndenumerate(table):
        if np.isfinite(value):
            text, color = f"{value:.4f}", "white" if c_norm(value) > 0.6 else "#1f1e1d"
        else:
            text, color = "div.", "#1f1e1d"
        ax.text(j, i, text, ha="center", va="center", fontsize=6, color=color)
    ax.set_xticks(range(len(N_NODES_PER_LAYER)), N_NODES_PER_LAYER)
    ax.set_yticks(range(len(N_HIDDEN_LAYERS)), N_HIDDEN_LAYERS)
    ax.set_xlabel("Nodes per hidden layer")
    ax.set_title(activation, fontsize=9)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
axes[0].set_ylabel("Hidden layers")
fig.colorbar(image, ax=axes, label="Cross-validated MSE", shrink=0.9)
fig.savefig("../docs/figures/regression/activation_architectures.pdf")
plt.show()

## Overfitting

### Model size

The training MSE (dashed) and the cross-validated MSE (solid) as functions of the number of weights and biases, one line per depth. A widening gap between the two, with the training MSE below the noise floor $\sigma^2$, means the network fits the noise in its training folds rather than the Runge function. Diverged architectures are left out of the lines.

In [ ]:
# One sequential hue, light to dark with increasing depth
depth_colors = dict(
    zip(
        N_HIDDEN_LAYERS,
        plt.get_cmap("Blues")(np.linspace(0.4, 1.0, len(N_HIDDEN_LAYERS))),
        strict=True,
    )
)

fig, axes = plt.subplots(
    1, len(ACTIVATIONS_BY_NAME), figsize=(7, 3), sharey=True, constrained_layout=True
)
for ax, activation in zip(axes, ACTIVATIONS_BY_NAME, strict=True):
    for n_layers in N_HIDDEN_LAYERS:
        rows = c_best[(c_best["activation"] == activation) & (c_best["n_layers"] == n_layers)]
        rows = rows.sort_values("n_parameters")
        color = depth_colors[n_layers]
        ax.plot(rows["n_parameters"], rows["cv_mse"], lw=1.2, ls="-", marker="o", ms=3, color=color)
        ax.plot(rows["n_parameters"], rows["train_mse"], lw=1.2, ls="--", color=color)
    ax.axhline(NOISE_STD**2, color="black", ls=":", lw=0.8)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(activation, fontsize=9)
    ax.set_xlabel("Number of parameters")
    ax.grid(alpha=0.3, lw=0.5)
axes[0].set_ylabel("MSE")
handles = [
    Line2D(
        [],
        [],
        color=depth_colors[n_layers],
        lw=1.2,
        label=f"{n_layers} hidden layer{'s' if n_layers > 1 else ''}",
    )
    for n_layers in N_HIDDEN_LAYERS
] + [
    Line2D([], [], color="#5f5e5a", lw=1.2, marker="o", ms=3, label="validation (CV)"),
    Line2D([], [], color="#5f5e5a", lw=1.2, ls="--", label="training"),
    Line2D([], [], color="black", lw=0.8, ls=":", label=r"noise floor $\sigma^2$"),
]
fig.legend(handles=handles, loc="outside lower center", ncols=4, frameon=False, fontsize=8)
fig.savefig("../docs/figures/regression/activation_overfitting.pdf")
plt.show()

### Overfitting during training

Overfitting also develops with the number of epochs. We hold out one fifth of the training data as a validation set, train the smallest and the largest architecture for each activation (each with its best learning rate) for longer than in the search, and record the MSE on both parts after every epoch with `fit`'s `validation_data`. A validation curve that turns upward while the training curve keeps falling below $\sigma^2$ is the network fitting the noise. The vertical axis is cut at an MSE of 1, so the tops of short spikes, where a minibatch step throws the network far off before it recovers, are not shown.

In [ ]:
CURVE_SIZES = [(1, min(N_NODES_PER_LAYER)), (max(N_HIDDEN_LAYERS), max(N_NODES_PER_LAYER))]

# One held-out fold of the training data, standardized on the remaining part
x_fit, x_val, y_fit, y_val = train_test_split(
    x_train, y_train, test_size=1 / N_FOLDS, random_state=SEED
)
curve_scaler = StandardScaler().fit(x_fit.reshape(-1, 1))
X_fit = curve_scaler.transform(x_fit.reshape(-1, 1))
X_val = curve_scaler.transform(x_val.reshape(-1, 1))

# Upper limit of the plot, so that short spikes do not hide the rest of the curves
C_CURVE_MSE_MAX = 1.0

best_learning_rate = c_best.set_index(["activation", "n_layers", "n_nodes"])["learning_rate"]


def train_val_curves(
    activation: str, n_layers: int, n_nodes: int, learning_rate: float
) -> tuple[list[float], list[float]]:
    """Training and validation MSE per epoch on the held-out split of the training data."""
    network = create_ffnn(
        hidden_layers=[n_nodes] * n_layers,
        optimizer=C_OPTIMIZER(learning_rate=learning_rate),
        activation=ACTIVATIONS_BY_NAME[activation],
        rng=np.random.default_rng(SEED),
    )
    with np.errstate(over="ignore", invalid="ignore"):
        train_history = network.fit(
            X_fit,
            y_fit,
            epochs=C_CURVE_EPOCHS,
            batch_size=C_BATCH_SIZE,
            validation_data=(X_val, y_val),
        )
    return train_history, network.validation_history


c_curve_configs = list(product(ACTIVATIONS_BY_NAME, CURVE_SIZES))
c_curves = dict(
    zip(
        c_curve_configs,
        Parallel(n_jobs=-1)(
            delayed(train_val_curves)(
                activation, n_layers, n_nodes, best_learning_rate[activation, n_layers, n_nodes]
            )
            for activation, (n_layers, n_nodes) in c_curve_configs
        ),
        strict=True,
    )
)

size_colors = dict(
    zip(CURVE_SIZES, plt.get_cmap("Blues")(np.linspace(0.5, 1.0, len(CURVE_SIZES))), strict=True)
)

fig, axes = plt.subplots(
    1, len(ACTIVATIONS_BY_NAME), figsize=(7, 3), sharey=True, constrained_layout=True
)
for ax, activation in zip(axes, ACTIVATIONS_BY_NAME, strict=True):
    for size in CURVE_SIZES:
        train_history, val_history = c_curves[activation, size]
        epochs = np.arange(1, len(train_history) + 1)
        ax.plot(epochs, val_history, lw=1.2, ls="-", color=size_colors[size])
        ax.plot(epochs, train_history, lw=1.2, ls="--", color=size_colors[size])
    ax.axhline(NOISE_STD**2, color="black", ls=":", lw=0.8)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_ylim(top=C_CURVE_MSE_MAX)
    ax.set_title(activation, fontsize=9)
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.3, lw=0.5)
axes[0].set_ylabel("MSE")
handles = [
    Line2D([], [], color=size_colors[size], lw=1.2, label=architecture_label(*size))
    for size in CURVE_SIZES
] + [
    Line2D([], [], color="#5f5e5a", lw=1.2, label="validation"),
    Line2D([], [], color="#5f5e5a", lw=1.2, ls="--", label="training"),
    Line2D([], [], color="black", lw=0.8, ls=":", label=r"noise floor $\sigma^2$"),
]
fig.legend(
    handles=handles, loc="outside lower center", ncols=len(handles), frameon=False, fontsize=8
)
fig.savefig("../docs/figures/regression/activation_curves.pdf")
plt.show()

## Comparison of the selected networks

For each activation, we retrain the architecture with the lowest cross-validated MSE on all the training data, and compare it with OLS and the sigmoid networks from part b using the same metrics as above.

In [ ]:
c_networks: dict[str, FFNN] = {}
C_MODEL_LABELS = dict(MODEL_LABELS)
for activation, config in best_per_activation.iterrows():
    n_layers, n_nodes = int(config.n_layers), int(config.n_nodes)
    network = create_ffnn(
        hidden_layers=[n_nodes] * n_layers,
        optimizer=C_OPTIMIZER(learning_rate=config.learning_rate),
        activation=ACTIVATIONS_BY_NAME[activation],
        rng=np.random.default_rng(SEED),
    )
    network.fit(X_train, y_train, epochs=C_EPOCHS, batch_size=C_BATCH_SIZE)
    c_networks[activation] = network
    C_MODEL_LABELS[activation] = f"FFNN, {activation} ({architecture_label(n_layers, n_nodes)})"

c_fitted = fitted | {
    activation: lambda x_values, net=network: net.predict(scaler.transform(x_values.reshape(-1, 1)))
    for activation, network in c_networks.items()
}

fig, ax = plt.subplots(figsize=(6, 4.6), constrained_layout=True)
ax.scatter(x_train, y_train, s=10, color="#b4b2a9", label="training data")
ax.scatter(
    x_test, y_test, s=14, facecolors="none", edgecolors="#5f5e5a", linewidths=0.8, label="test data"
)
ax.plot(x_plot, runge_function(x_plot), color="black", lw=1.5, label="Runge function")
for activation in c_networks:
    ax.plot(x_plot, c_fitted[activation](x_plot), lw=1.5, label=C_MODEL_LABELS[activation])
ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.grid(alpha=0.3, lw=0.5)
fig.legend(loc="outside lower center", ncols=2, frameon=False, fontsize=8)
fig.savefig("../docs/figures/regression/activation_comparison.pdf")
plt.show()

c_metrics = pd.DataFrame(
    {
        C_MODEL_LABELS[name]: {
            "train MSE": mean_squared_error(y_train, predict(x_train)),
            "test MSE": mean_squared_error(y_test, predict(x_test)),
            "train R2": r2_score(y_train, predict(x_train)),
            "test R2": r2_score(y_test, predict(x_test)),
            "MSE vs f": mean_squared_error(runge_function(x_plot), predict(x_plot)),
        }
        for name, predict in c_fitted.items()
    }
).T
c_metrics.round(4)

# Part d) Testing different norms

We now add the hyperparamters $\lambda$ with the $\ell_1$ and $\ell_2$ norms and find the optimal results and compare these with the Ridge and Lasso regression models we found in Project 1. We use the exact same data sets as in the previous project.

The `FFNN` class takes in `regularization` (`"none"`, `"l1"` or `"l2"`) and `lambda_` ($\geq 0$) in its constructor, which we can use for applying the regularization.

First, we define the hyperparameter space for which we will search:

In [ ]:
D_REGULARIZATION = ["l1", "l2"]
D_REGULARIZATION_STRENGTH = [0.1, 0.01, 0.001, 0.0001, 0.00001, 0.0]

D_N_HIDDEN_LAYERS = [4, 8]
D_N_NODES_PER_LAYER = [40, 80]
D_ACTIVATION = [Sigmoid, ReLU, LeakyReLU]

# The training setup is that of part c (C_OPTIMIZER, C_BATCH_SIZE, C_EPOCHS), so that
# lambda = 0 reproduces the unregularized networks there
D_LEARNING_RATE = [0.01, 0.001, 0.0001]

# Cached regularization search, recomputed under the same rules as GRID_SEARCH_PATH
D_GRID_SEARCH_PATH = Path("../data/results/d_regularization_search.csv")

## Regularization search

We search over the norm, the strength $\lambda$ and the learning rate $\gamma$ for the deeper and wider architectures, since those overfit most in part c and have the most to gain from a penalty. Every configuration is scored with the same 5-fold cross-validation, training setup and initial weights as in part c, through `cv_train_val_mse`. The score is the MSE of the predictions, without the penalty, so that networks with different $\lambda$ are compared on equal terms. $\lambda = 0$ is the unregularized baseline; it is trained once per norm, giving identical results for both. The search caches its results in `D_GRID_SEARCH_PATH`.

In [ ]:
D_GRID_COLUMNS = ["activation", "n_layers", "n_nodes", "regularization", "lambda_", "learning_rate"]

d_grid = pd.DataFrame(
    product(
        [activation.__name__ for activation in D_ACTIVATION],
        D_N_HIDDEN_LAYERS,
        D_N_NODES_PER_LAYER,
        D_REGULARIZATION,
        D_REGULARIZATION_STRENGTH,
        D_LEARNING_RATE,
    ),
    columns=D_GRID_COLUMNS,
)

d_cached = pd.read_csv(D_GRID_SEARCH_PATH) if D_GRID_SEARCH_PATH.exists() else None
if d_cached is not None and d_cached[D_GRID_COLUMNS].equals(d_grid):
    d_results = d_cached
else:
    d_scores = Parallel(n_jobs=-1, verbose=1)(
        delayed(cv_train_val_mse)(
            row.activation,
            int(row.n_layers),
            int(row.n_nodes),
            row.learning_rate,
            regularization=row.regularization,
            lambda_=row.lambda_,
        )
        for row in d_grid.itertuples(index=False)
    )
    d_results = d_grid.assign(
        train_mse=[train for train, _ in d_scores], cv_mse=[val for _, val in d_scores]
    )
    D_GRID_SEARCH_PATH.parent.mkdir(parents=True, exist_ok=True)
    d_results.to_csv(D_GRID_SEARCH_PATH, index=False)

The best configuration for each activation and norm, with the cross-validated MSE of the same architecture without regularization ($\lambda = 0$, at its best learning rate) for comparison. If the best $\lambda$ is 0, the penalty did not help for that activation and norm.

In [ ]:
# The best learning rate for each architecture, norm and strength
d_best = d_results.loc[
    d_results.groupby(["activation", "n_layers", "n_nodes", "regularization", "lambda_"])[
        "cv_mse"
    ].idxmin()
]

# The unregularized baseline of each architecture, identical for both norms
d_unregularized = (
    d_best[d_best["lambda_"] == 0]
    .groupby(["activation", "n_layers", "n_nodes"])["cv_mse"]
    .min()
    .rename("cv_mse_unregularized")
)

best_per_norm = (
    d_best.loc[d_best.groupby(["activation", "regularization"])["cv_mse"].idxmin()]
    .join(d_unregularized, on=["activation", "n_layers", "n_nodes"])
    .set_index(["activation", "regularization"])
)
best_per_norm.round({"train_mse": 5, "cv_mse": 5, "cv_mse_unregularized": 5})

## Strength of the penalty

The training MSE (dashed) and the cross-validated MSE (solid) as functions of $\lambda$, one line per architecture, each at its best learning rate. $\lambda = 0$ cannot be shown on a logarithmic axis, so it is drawn one decade below the smallest $\lambda$ and labelled 0. Without a penalty the training MSE falls below the noise floor $\sigma^2$ while the validation MSE stays above it: the network fits the noise. A good $\lambda$ raises the training MSE towards $\sigma^2$ and lowers the validation MSE, while a too large $\lambda$ shrinks the weights until the network predicts the mean of the data, with an MSE of about $\mathrm{Var}(y) \approx 0.1$.

In [ ]:
D_ACTIVATION_NAMES = [activation.__name__ for activation in D_ACTIVATION]
D_ARCHITECTURES = list(product(D_N_HIDDEN_LAYERS, D_N_NODES_PER_LAYER))
NORM_LABELS = {"l1": "$L_1$", "l2": "$L_2$"}

lambdas = sorted(D_REGULARIZATION_STRENGTH)
# lambda = 0 is drawn one decade below the smallest positive lambda on the log axis
lambda_zero_position = min(lam for lam in lambdas if lam > 0) / 10
lambda_positions = [lam if lam > 0 else lambda_zero_position for lam in lambdas]
lambda_labels = ["0" if lam == 0 else f"$10^{{{round(np.log10(lam))}}}$" for lam in lambdas]

# One sequential hue, light to dark with increasing number of parameters
architecture_colors = dict(
    zip(
        D_ARCHITECTURES,
        plt.get_cmap("Blues")(np.linspace(0.4, 1.0, len(D_ARCHITECTURES))),
        strict=True,
    )
)

fig, axes = plt.subplots(
    len(D_REGULARIZATION),
    len(D_ACTIVATION_NAMES),
    figsize=(7, 4.6),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
for (i, regularization), (j, activation) in product(
    enumerate(D_REGULARIZATION), enumerate(D_ACTIVATION_NAMES)
):
    ax = axes[i, j]
    for n_layers, n_nodes in D_ARCHITECTURES:
        rows = d_best[
            (d_best["activation"] == activation)
            & (d_best["regularization"] == regularization)
            & (d_best["n_layers"] == n_layers)
            & (d_best["n_nodes"] == n_nodes)
        ].sort_values("lambda_")
        color = architecture_colors[n_layers, n_nodes]
        ax.plot(lambda_positions, rows["cv_mse"], lw=1.2, ls="-", marker="o", ms=3, color=color)
        ax.plot(lambda_positions, rows["train_mse"], lw=1.2, ls="--", color=color)
    ax.axhline(NOISE_STD**2, color="black", ls=":", lw=0.8)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xticks(lambda_positions, lambda_labels)
    ax.minorticks_off()
    ax.set_title(f"{activation}, {NORM_LABELS[regularization]}", fontsize=9)
    ax.grid(alpha=0.3, lw=0.5)
for ax in axes[-1]:
    ax.set_xlabel(r"$\lambda$")
for ax in axes[:, 0]:
    ax.set_ylabel("MSE")
handles = [
    Line2D([], [], color=architecture_colors[size], lw=1.2, label=architecture_label(*size))
    for size in D_ARCHITECTURES
] + [
    Line2D([], [], color="#5f5e5a", lw=1.2, marker="o", ms=3, label="validation (CV)"),
    Line2D([], [], color="#5f5e5a", lw=1.2, ls="--", label="training"),
    Line2D([], [], color="black", lw=0.8, ls=":", label=r"noise floor $\sigma^2$"),
]
fig.legend(handles=handles, loc="outside lower center", ncols=4, frameon=False, fontsize=8)
fig.savefig("../docs/figures/regression/regularization_lambda.pdf")
plt.show()

## Penalty and learning rate

The cross-validated MSE over $\lambda$ and the learning rate $\gamma$, for the architecture of the best configuration of each activation and norm (the table above). The framed cell is the minimum. A cell is marked "div." if training diverged.

In [ ]:
d_learning_rates = sorted(D_LEARNING_RATE)

d_lr_tables = {}
for (activation, regularization), config in best_per_norm.iterrows():
    rows = d_results[
        (d_results["activation"] == activation)
        & (d_results["regularization"] == regularization)
        & (d_results["n_layers"] == config.n_layers)
        & (d_results["n_nodes"] == config.n_nodes)
    ]
    d_lr_tables[activation, regularization] = (
        rows.pivot_table(index="lambda_", columns="learning_rate", values="cv_mse")
        .loc[lambdas, d_learning_rates]
        .to_numpy()
    )

d_finite = np.concatenate([table[np.isfinite(table)] for table in d_lr_tables.values()])
d_norm = LogNorm(vmin=d_finite.min(), vmax=d_finite.max())

fig, axes = plt.subplots(
    len(D_REGULARIZATION),
    len(D_ACTIVATION_NAMES),
    figsize=(7, 5.4),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
for (i, regularization), (j, activation) in product(
    enumerate(D_REGULARIZATION), enumerate(D_ACTIVATION_NAMES)
):
    ax = axes[i, j]
    table = d_lr_tables[activation, regularization]
    image = ax.imshow(np.ma.masked_invalid(table), cmap=cmap, norm=d_norm, aspect="auto")
    for (row, col), value in np.ndenumerate(table):
        if np.isfinite(value):
            text, color = f"{value:.4f}", "white" if d_norm(value) > 0.6 else "#1f1e1d"
        else:
            text, color = "div.", "#1f1e1d"
        ax.text(col, row, text, ha="center", va="center", fontsize=6, color=color)
    best_row, best_col = np.unravel_index(np.argmin(table), table.shape)
    ax.add_patch(
        Rectangle((best_col - 0.5, best_row - 0.5), 1, 1, fill=False, ec="#d85a30", lw=1.5)
    )
    config = best_per_norm.loc[activation, regularization]
    architecture = architecture_label(int(config.n_layers), int(config.n_nodes))
    ax.set_title(f"{activation} ({architecture}), {NORM_LABELS[regularization]}", fontsize=8)
    ax.set_xticks(
        range(len(d_learning_rates)),
        [f"$10^{{{round(np.log10(lr))}}}$" for lr in d_learning_rates],
    )
    ax.set_yticks(range(len(lambdas)), lambda_labels)
    ax.tick_params(which="both", length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
for ax in axes[-1]:
    ax.set_xlabel(r"Learning rate $\gamma$")
for ax in axes[:, 0]:
    ax.set_ylabel(r"$\lambda$")
fig.colorbar(image, ax=axes, label="Cross-validated MSE", shrink=0.9)
fig.savefig("../docs/figures/regression/regularization_heatmaps.pdf")
plt.show()

## Comparison with Ridge and Lasso from project 1

We compare the network with the lowest cross-validated MSE for each norm (over all activations and architectures in the search) with the best Ridge ($L_2$) and Lasso ($L_1$) models from project 1. There, 5-fold CV over the polynomial degree and $\lambda$ on the same training data selected Ridge with $p = 11$ and $\lambda = 10^{-6}$, and Lasso with $p = 6$ and $\lambda = 10^{-2.5}$, fitted with Adam ($\gamma = 0.05$, 5000 iterations). We refit both on the training data, with the polynomial features standardized as for OLS above, and retrain the networks on all the training data with their selected $\lambda$ and $\gamma$. All models are thus selected with 5-fold CV on the same 80 samples, and the 20 test samples are used only here.

Project 1 found that our Lasso solver does not converge at high polynomial degrees, which pushed its selection towards a low degree and a large penalty, so the Lasso model is weaker than the $L_1$ penalty itself would allow. The test set has only 20 samples, so the MSE against $f$ is the more reliable number for the comparison.

In [ ]:
# The best Ridge and Lasso models of project 1, selected with 5-fold CV on the training data
RIDGE_DEGREE, RIDGE_LAMBDA = 11, 1e-6
LASSO_DEGREE, LASSO_LAMBDA = 6, 10**-2.5
LASSO_SOLVER = {"learning_rate": 0.05, "optimizer": "adam", "max_iter": 5000}


def fit_polynomial_model(model: OLS | Ridge | Lasso, degree: int):
    """Fit a linear model on standardized polynomial features of the training data.

    Returns:
        The fitted model as a function of the raw x.
    """
    poly_scaler = StandardScaler().fit(univariate_polynomial_design_matrix(x_train, degree))
    model.fit(poly_scaler.transform(univariate_polynomial_design_matrix(x_train, degree)), y_train)
    return lambda x_values: model.predict(
        poly_scaler.transform(univariate_polynomial_design_matrix(x_values, degree))
    )


def lambda_label(lam: float) -> str:
    """Penalty as a power of ten, e.g. 10^{-2.5}."""
    return rf"$\lambda = 10^{{{np.log10(lam):g}}}$"


d_fitted = {
    "Ridge": fit_polynomial_model(Ridge(lam=RIDGE_LAMBDA), RIDGE_DEGREE),
    "Lasso": fit_polynomial_model(Lasso(lam=LASSO_LAMBDA, **LASSO_SOLVER), LASSO_DEGREE),
}
D_MODEL_LABELS = {
    "Ridge": f"Ridge ($p = {RIDGE_DEGREE}$, {lambda_label(RIDGE_LAMBDA)})",
    "Lasso": f"Lasso ($p = {LASSO_DEGREE}$, {lambda_label(LASSO_LAMBDA)})",
}

# The network with the lowest cross-validated MSE for each norm
d_selected = (
    best_per_norm.reset_index()
    .loc[lambda df: df.groupby("regularization")["cv_mse"].idxmin()]
    .set_index("regularization")
)
for regularization, config in d_selected.iterrows():
    n_layers, n_nodes = int(config.n_layers), int(config.n_nodes)
    network = create_ffnn(
        hidden_layers=[n_nodes] * n_layers,
        optimizer=C_OPTIMIZER(learning_rate=config.learning_rate),
        activation=ACTIVATIONS_BY_NAME[config.activation],
        regularization=regularization,
        lambda_=config.lambda_,
        rng=np.random.default_rng(SEED),
    )
    network.fit(X_train, y_train, epochs=C_EPOCHS, batch_size=C_BATCH_SIZE)
    d_fitted[regularization] = lambda x_values, net=network: net.predict(
        scaler.transform(x_values.reshape(-1, 1))
    )
    D_MODEL_LABELS[regularization] = (
        f"FFNN, {config.activation} ({architecture_label(n_layers, n_nodes)}), "
        f"{NORM_LABELS[regularization]}, {lambda_label(config.lambda_)}"
    )

# Each penalty next to its linear counterpart
D_PANELS = {"$L_2$ penalty": ["Ridge", "l2"], "$L_1$ penalty": ["Lasso", "l1"]}

fig, axes = plt.subplots(1, len(D_PANELS), figsize=(7, 4.4), sharey=True, constrained_layout=True)
for ax, (title, names) in zip(axes, D_PANELS.items(), strict=True):
    ax.scatter(x_train, y_train, s=10, color="#b4b2a9", label="training data")
    ax.scatter(
        x_test,
        y_test,
        s=14,
        facecolors="none",
        edgecolors="#5f5e5a",
        linewidths=0.8,
        label="test data",
    )
    ax.plot(x_plot, runge_function(x_plot), color="black", lw=1.5, label="Runge function")
    for name in names:
        ax.plot(x_plot, d_fitted[name](x_plot), lw=1.5, label=D_MODEL_LABELS[name])
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("$x$")
    ax.grid(alpha=0.3, lw=0.5)
    ax.legend(frameon=False, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.16))
axes[0].set_ylabel("$y$")
fig.savefig("../docs/figures/regression/regularization_comparison.pdf")
plt.show()

d_metrics = pd.DataFrame(
    {
        D_MODEL_LABELS[name]: {
            "train MSE": mean_squared_error(y_train, d_fitted[name](x_train)),
            "test MSE": mean_squared_error(y_test, d_fitted[name](x_test)),
            "train R2": r2_score(y_train, d_fitted[name](x_train)),
            "test R2": r2_score(y_test, d_fitted[name](x_test)),
            "MSE vs f": mean_squared_error(runge_function(x_plot), d_fitted[name](x_plot)),
        }
        for name in ["Ridge", "l2", "Lasso", "l1"]
    }
).T
d_metrics.round(4)